# 07 — Predict full test_users + diversify + submission

1. Sinh candidate cho **toàn bộ test_users** (cùng pipeline notebook 04, dùng FULL train data).
2. Build feature matrix với cutoff = TRAIN_END.
3. Predict score → top-30 / user.
4. Diversify (max_per_seller=7, max_per_district=8, freshness_boost).
5. Submission validator + write csv.

In [ ]:
# Local kernel: assume deps already installed.
# To install run once:
#   pip install pyarrow pandas numpy scipy lightgbm scikit-learn tqdm
print("Skipping pip install (local kernel).")

In [ ]:
# === Local setup: paths + add training/utils to sys.path ===
import os, sys

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
DATA_DIR     = os.path.join(PROJECT_DIR, "Datathon_Data")  # contains train/ and test/
os.makedirs(CACHE_DIR, exist_ok=True)

if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("CACHE_DIR  :", CACHE_DIR)
print("utils available:", os.path.isdir(os.path.join(TRAINING_DIR, "utils")))

In [ ]:
TRAIN_DATE_START = "2025-11-09"
TRAIN_DATE_END = "2026-04-09"
VALID_START = "2026-03-13"

POSITIVE_EVENT_TYPES = frozenset({
    "view_phone", "contact_chat", "other_interaction", "contact_zalo", "contact_sms",
})
HIGH_INTENT_EVENTS = frozenset({"view_phone", "contact_chat", "contact_zalo", "contact_sms"})

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}

# Local data paths (relative to DATA_DIR defined in the setup cell)
TRAIN_PATH = os.path.join(DATA_DIR, "train") + os.sep
TEST_PATH  = os.path.join(DATA_DIR, "test", "test") + os.sep

print("Constants loaded. TRAIN_END:", TRAIN_DATE_END, "VALID_START:", VALID_START)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

In [ ]:
# Local mode: no GCS egress guardrail needed.
print("Local kernel: reading from CACHE_DIR.")

In [ ]:
import time, os
import numpy as np
import pandas as pd
import lightgbm as lgb

from utils.covis import build_covis
from utils.candidates import (
    gen_history_candidates, gen_covis_candidates,
    gen_popularity_candidates, gen_content_candidates,
    merge_candidates,
)
from utils.features import build_user_features, build_item_features, add_cross_features
from utils.diversify import diversify_top_k
from utils.submit import validate_submission, write_submission

t0 = time.time()
events_pos = pd.read_parquet(f"{CACHE_DIR}/events_positive.parquet")
events_pos["event_ts"] = pd.to_datetime(events_pos["event_ts"])
enr = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet")
test_users = pd.read_parquet(f"{CACHE_DIR}/test_users.parquet")
print(f"events_pos: {len(events_pos):,} | enr: {len(enr):,} | "
      f"test_users: {len(test_users):,} | {time.time()-t0:.1f}s")

TEST_UIDS = set(test_users["user_id"].unique())
events_pos_test = events_pos[events_pos["user_id"].isin(TEST_UIDS)].copy()
print(f"events_pos for test users: {len(events_pos_test):,}")

In [ ]:
# ---- Candidates (4 sources) for ALL test_users ----
allowed_a = set(enr[enr["tier"] == "A"]["item_id"])
allowed_ab = set(enr[enr["tier"].isin(["A", "B"])]["item_id"])

t0 = time.time()
hist_cands = gen_history_candidates(events_pos_test, allowed_ab, top_n=30)
print(f"history: {sum(len(v) for v in hist_cands.values()):,} | {time.time()-t0:.1f}s")

t0 = time.time()
covis_input = events_pos[events_pos["item_id"].isin(allowed_a)][
    ["user_id", "session_id", "item_id", "event_type", "event_ts"]
]
covis = build_covis(covis_input, allowed_items=allowed_a,
                    top_k_per_item=20, time_decay=True)
covis_cands = gen_covis_candidates(events_pos_test, covis,
                                    allowed_items=allowed_a, top_n=200)
print(f"covis: {sum(len(v) for v in covis_cands.values()):,} | {time.time()-t0:.1f}s")

In [ ]:
# user_profile for popularity + content
def _mode(s):
    s = s.dropna()
    return s.mode().iloc[0] if len(s) else None

enr_seg_cols = [c for c in ["item_id", "category", "city_name",
                             "district_name", "ad_type"] if c in enr.columns]
tp_t = events_pos_test.merge(enr[enr_seg_cols], on="item_id", how="left")

prof_specs = {}
SEG_MAP = [("category", "u_top_category"), ("city_name", "u_top_city"),
            ("district_name", "u_top_district"), ("ad_type", "u_top_ad_type")]
for src, out in SEG_MAP:
    if src in tp_t.columns:
        prof_specs[out] = (src, _mode)
user_profile = tp_t.groupby("user_id").agg(**prof_specs).reset_index()

# For cold test_users (no history), fill with global mode
missing = TEST_UIDS - set(user_profile["user_id"])
print(f"Cold test users (no history): {len(missing):,}")
if missing:
    cold_row = {"user_id": list(missing)}
    for src, out in SEG_MAP:
        if src in tp_t.columns:
            m = tp_t[src].mode()
            cold_row[out] = m.iloc[0] if len(m) else None
    user_profile = pd.concat([user_profile, pd.DataFrame(cold_row)], ignore_index=True)

snap = pd.read_parquet(f"{CACHE_DIR}/snapshot_60d.parquet")
snap["date"] = pd.to_datetime(snap["date"])
last14_start = pd.Timestamp(TRAIN_DATE_END) - pd.Timedelta(days=14)
snap_14 = snap[snap["date"] >= last14_start]

pop_cands = gen_popularity_candidates(user_profile, enr, snap_14,
                                       top_n_per_seg=50, top_n_per_user=100)
content_cands = gen_content_candidates(user_profile, enr, top_n=50)
print(f"pop: {sum(len(v) for v in pop_cands.values()):,} | "
      f"content: {sum(len(v) for v in content_cands.values()):,}")

In [ ]:
# Merge candidates
cands = merge_candidates({
    "history": hist_cands,
    "covis": covis_cands,
    "pop": pop_cands,
    "content": content_cands,
}, cap_total=500)
print(f"candidates total: {len(cands):,} | users: {cands['user_id'].nunique():,}")

# Ensure every test user has at least global pop fallback
missing_uids = TEST_UIDS - set(cands["user_id"])
print(f"Users still missing: {len(missing_uids):,}")
if missing_uids:
    pop_global = (snap_14.groupby("item_id")["contacts_24h"].sum()
                  .sort_values(ascending=False).head(100).index.tolist())
    pop_global = [it for it in pop_global if it in set(enr[enr["tier"].isin(["A","B"])]["item_id"])]
    fb_rows = []
    for u in missing_uids:
        for it in pop_global[:100]:
            fb_rows.append({"user_id": u, "item_id": it,
                            "src_history": np.nan, "src_covis": np.nan,
                            "src_pop": 1.0, "src_content": np.nan})
    cands = pd.concat([cands, pd.DataFrame(fb_rows)], ignore_index=True)
print(f"After fallback: users {cands['user_id'].nunique():,}")
# ---- Filter out purchased listings (user-item pairs where purchased=True) ----
_pci = pd.read_parquet(f"{CACHE_DIR}/pci_full.parquet", columns=["user_id", "item_id", "purchased"])
_purchased = _pci[_pci["purchased"] == True][["user_id", "item_id"]].drop_duplicates()
_purchased["_drop"] = True
_before = len(cands)
cands = cands.merge(_purchased, on=["user_id", "item_id"], how="left")
cands = cands[cands["_drop"].isna()].drop(columns=["_drop"])
print(f"Purchased filter: {_before - len(cands):,} rows removed -> {len(cands):,} remain")
del _pci, _purchased


In [ ]:
# Build features at cutoff = TRAIN_END (use full events_pos, not just train_pos)
CUTOFF = pd.Timestamp(TRAIN_DATE_END) + pd.Timedelta(seconds=1)
pv_path = f"{CACHE_DIR}/events_pageview_30d.parquet"
if os.path.exists(pv_path):
    pv = pd.read_parquet(pv_path)
    pv["event_ts"] = pd.to_datetime(pv["event_ts"])
else:
    pv = pd.DataFrame(columns=["user_id", "item_id", "event_ts", "dwell_time_sec"])

user_feats = build_user_features(events_pos, pv, cutoff_ts=CUTOFF, dim_enriched=enr)
item_feats = build_item_features(events_pos, snap, enr, cutoff_ts=CUTOFF)
print(f"user_feats: {user_feats.shape} | item_feats: {item_feats.shape}")

full = add_cross_features(cands, user_feats, item_feats)
print(f"test matrix: {full.shape}")

In [ ]:
# Load trained model — prefer tuned model if available
_tuned = f"{CACHE_DIR}/model_tuned.txt"
_base  = f"{CACHE_DIR}/model.txt"
model_path = _tuned if os.path.exists(_tuned) else _base
print(f"Using model: {model_path}")
model = lgb.Booster(model_file=model_path)

# Align features with trained model
trained_feats = model.feature_name()
for c in trained_feats:
    if c not in full.columns:
        full[c] = np.nan
X = full[trained_feats].copy()
for c in X.columns:
    if X[c].dtype == "object":
        X[c] = X[c].astype("category")

t0 = time.time()
full["_pred"] = model.predict(X)
print(f"predicted in {time.time()-t0:.0f}s")

In [ ]:
# Top-30 per user → diversify → top-10
top30 = (full.sort_values(["user_id", "_pred"], ascending=[True, False])
         .groupby("user_id").head(30)[["user_id", "item_id", "_pred"]]
         .rename(columns={"_pred": "score"}))
print(f"top30: {len(top30):,}")

dim_lookup = enr[["item_id", "seller_id", "district_name"]].copy()
if "i_listing_age_days_latest" in item_feats.columns:
    dim_lookup = dim_lookup.merge(
        item_feats[["item_id", "i_listing_age_days_latest"]],
        on="item_id", how="left",
    )

top10 = diversify_top_k(top30, dim_lookup, k=10,
                         max_per_seller=7, max_per_district=8,
                         freshness_boost=0.05, fresh_age_days=7)
print(f"top10 after diversify: {len(top10):,} | users: {top10['user_id'].nunique():,}")

In [ ]:
# Validate + write submission
valid_items = set(enr["item_id"])
validate_submission(top10, valid_items, TEST_UIDS, k=10)
out = f"{CACHE_DIR}/submission.csv"
write_submission(top10, out)
print(f"\nSubmission ready: {out}")
print(f"File size: {os.path.getsize(out)/(1024**2):.2f} MB")